# Create a global barren area mask for ERA5-Drought

## About this jupyter notebook

This jupyter notebook creates a global barren area mask for ERA5-Drought, using ERA5 data downloaded from the Climate Data Store (CDS). 

We define barren areas as follows:
every pixel with less than 50% land (land fraction < 0.5, parameter `min_landfrac` below), no high or low vegetation (low vegetation cover fraction = 0, high vegetation cover fraction = 0) or a snow depth larger than 10 m as an indicator for glaciers (snow depth >= 10, parameter `snow_depth` below) and where the annual mean precipitation for 1991–2020 (i.e., the WMO reference period) is less than 0.3 mm day-1 (parameter `pmin` below) is masked out.

We proceed in two steps below:
1. Download all relevant ERA5 Data
2. Create a barren area mask and save as a netcdf file


## Libraries

In [1]:
import os 
import cdsapi
import numpy as np
import xarray as xr
import regionmask
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import datetime as dt
from datetime import UTC
from calendar import monthrange

## Settings

In [2]:
min_landfrac = 0.5    # unit: - 
snow_depth   = 10     # unit: m
pmin         = 0.0003 # unit: m

## 1. Download ERA5 data

We download two ERA5 datasets to create a barren mask: 

- Static vegetation data (from the _oper_ stream)
    - land-sea mask / land fraction
    - low vegetation cover
    - high vegetation cover
    - snow depth
- Average precipitation for all months in 1991-2020 (from the _moda_ stream, i.e. monthly mean of daily means)

We download the ERA5 data using the CDS API.

More info on how to use the CDS API can be found here: [https://cds.climate.copernicus.eu/how-to-api](https://cds.climate.copernicus.eu/how-to-api).

#### Static ERA5 data

In [ ]:
c = cdsapi.Client()
c.retrieve('reanalysis-era5-complete', { 
   'date'    : '2023-01-01/2023-02-01/2023-03-01/2023-04-01/2023-05-01/2023-06-01/2023-07-01/2023-08-01/2023-09-01/2023-10-01/2023-11-01/2023-12-01',
   'levtype' : 'sfc',
   'param'   : '27.128/28.128/141.128/172.128',     
    # Full information at https://apps.ecmwf.int/codes/grib/param-db/
    # We download:
    # cvl - var27 - low vegetation cover
    # cvh - var28 - high vegetation cover
    # sd  - var141 - snow depth (10m = glacier)
    # lsm - var172 - land-sea mask / land fraction
   'stream'  : 'oper',                  
   'expver'  : '1',                  
   'class'   : 'ea',                  
   'time'    : '00',                    
   'type'    : 'an',
   'area'    : '90/-180/-90/180',       # North, West, South, East. Default: global
   'grid'    : '0.25/0.25',             # Latitude/longitude. Default: spherical harmonics or reduced Gaussian grid
   'format'  : 'netcdf',                # Output needs to be regular lat-lon, so only works in combination with 'grid'!
}, 'era5_static_025x025.nc')            # Output file

2025-10-10 13:20:07,869 INFO Request ID is bdba51f6-ac73-4897-a4db-37f5961e4433
2025-10-10 13:20:07,924 INFO status has been updated to accepted


### Average rainfall

In [ ]:
c = cdsapi.Client()
c.retrieve('reanalysis-era5-complete', { 
    # unfortunately the date string needs to be messy for this stream
    # we download the full WMO reference period 1991–2020
   'date'  : "19910101/19910201/19910301/19910401/19910501/19910601/19910701/19910801/19910901/19911001/19911101/19911201/19920101/19920201/19920301/19920401/19920501/19920601/19920701/19920801/19920901/19921001/19921101/19921201/19930101/19930201/19930301/19930401/19930501/19930601/19930701/19930801/19930901/19931001/19931101/19931201/19940101/19940201/19940301/19940401/19940501/19940601/19940701/19940801/19940901/19941001/19941101/19941201/19950101/19950201/19950301/19950401/19950501/19950601/19950701/19950801/19950901/19951001/19951101/19951201/19960101/19960201/19960301/19960401/19960501/19960601/19960701/19960801/19960901/19961001/19961101/19961201/19970101/19970201/19970301/19970401/19970501/19970601/19970701/19970801/19970901/19971001/19971101/19971201/19980101/19980201/19980301/19980401/19980501/19980601/19980701/19980801/19980901/19981001/19981101/19981201/19990101/19990201/19990301/19990401/19990501/19990601/19990701/19990801/19990901/19991001/19991101/19991201/20000101/20000201/20000301/20000401/20000501/20000601/20000701/20000801/20000901/20001001/20001101/20001201/20010101/20010201/20010301/20010401/20010501/20010601/20010701/20010801/20010901/20011001/20011101/20011201/20020101/20020201/20020301/20020401/20020501/20020601/20020701/20020801/20020901/20021001/20021101/20021201/20030101/20030201/20030301/20030401/20030501/20030601/20030701/20030801/20030901/20031001/20031101/20031201/20040101/20040201/20040301/20040401/20040501/20040601/20040701/20040801/20040901/20041001/20041101/20041201/20050101/20050201/20050301/20050401/20050501/20050601/20050701/20050801/20050901/20051001/20051101/20051201/20060101/20060201/20060301/20060401/20060501/20060601/20060701/20060801/20060901/20061001/20061101/20061201/20070101/20070201/20070301/20070401/20070501/20070601/20070701/20070801/20070901/20071001/20071101/20071201/20080101/20080201/20080301/20080401/20080501/20080601/20080701/20080801/20080901/20081001/20081101/20081201/20090101/20090201/20090301/20090401/20090501/20090601/20090701/20090801/20090901/20091001/20091101/20091201/20100101/20100201/20100301/20100401/20100501/20100601/20100701/20100801/20100901/20101001/20101101/20101201/20110101/20110201/20110301/20110401/20110501/20110601/20110701/20110801/20110901/20111001/20111101/20111201/20120101/20120201/20120301/20120401/20120501/20120601/20120701/20120801/20120901/20121001/20121101/20121201/20130101/20130201/20130301/20130401/20130501/20130601/20130701/20130801/20130901/20131001/20131101/20131201/20140101/20140201/20140301/20140401/20140501/20140601/20140701/20140801/20140901/20141001/20141101/20141201/20150101/20150201/20150301/20150401/20150501/20150601/20150701/20150801/20150901/20151001/20151101/20151201/20160101/20160201/20160301/20160401/20160501/20160601/20160701/20160801/20160901/20161001/20161101/20161201/20170101/20170201/20170301/20170401/20170501/20170601/20170701/20170801/20170901/20171001/20171101/20171201/20180101/20180201/20180301/20180401/20180501/20180601/20180701/20180801/20180901/20181001/20181101/20181201/20190101/20190201/20190301/20190401/20190501/20190601/20190701/20190801/20190901/20191001/20191101/20191201/20200101/20200201/20200301/20200401/20200501/20200601/20200701/20200801/20200901/20201001/20201101/20201201",
   'levtype' : 'sfc',
   'param'   : '228.128',               # precipitation
   'stream'  : 'moda',                  # use 'moda' stream to reduce data request (moda: monthly means of daily means)
   'expver'  : '1',                  
   'class'   : 'ea',                  
   'time'    : '00',                    
   'type'    : 'fc',                    # precipitation comes from the forecast
   'area'    : '90/-180/-90/180',       
   'grid'    : '0.25/0.25',             
   'format'  : 'netcdf',                
}, 'era5_precip_025x025_ref.nc')        # Output file

## 2. Create the barren area mask

#### Read the static data

In [ ]:
vfile = f"era5_static_025x025.nc"
vdata = xr.open_dataset(vfile)
print(vdata)
# shift longitudes from 0/360 to -180/+180 if not retrieved on -180 to +180 grid
if vdata.coords['longitude'].max() > 300:
    vdata.coords['longitude'] = (vdata.coords['longitude'] + 180) % 360 - 180
    vdata = vdata.sortby(vdata.longitude)

### Read the precipitation data

In [ ]:
pfile = "era5_precip_025x025_ref.nc"
pdata = xr.open_dataset(pfile)
print(pdata)
# shift longitudes from 0/360 to -180/+180 if needed
if pdata.coords['longitude'].max() > 300:
    pdata.coords['longitude'] = (pdata.coords['longitude'] + 180) % 360 - 180
    pdata = pdata.sortby(pdata.longitude)

In [ ]:
# we calculate the exact daily precip, using all months during reference period from moda stream
# here multiplying with days in month and averaging 
# number of days in month
ndays = []
for itime in range(pdata['tp'].shape[0]):
    # date in file, convert from np.datetime64 to datetime.datetime
    fdate = dt.datetime.utcfromtimestamp(int(pdata.isel(valid_time=itime).valid_time.values)/1e9)
    # add 6h or 1day, because this is based on forecast and thus shifted by -6h
    idate = fdate + dt.timedelta(days=1)
    # get number of days in month
    ndays.append(monthrange(idate.year, idate.month)[1])
ndays_xr = xr.DataArray(ndays, dims=['valid_time'], coords=[pdata.valid_time])

# multiply with days in month to get total precipitation
pdata_mod = pdata * ndays_xr
# and now calculate sum over the reference period and divide by number of days
pdata_modmean = pdata_mod.sum(dim='valid_time') / ndays_xr.sum()
pdata_modmean['tp'].plot()

### Create the barren area mask

In [ ]:
# static criteria
mask = xr.where(vdata['lsm'].isel(valid_time=0) > min_landfrac, 1, 0)
mask = xr.where(vdata['sd'].isel(valid_time=0) >= snow_depth, 0, mask)
mask = xr.where( ((vdata['cvl'].isel(valid_time=0) == 0) & (vdata['cvh'].isel(valid_time=0) == 0)), 0, mask)
# precipitation criterion
mask = xr.where(pdata_modmean['tp'] < pmin, 0, mask)

In [ ]:
mask.plot()

### Write to netcdf file

In [ ]:
fmask = mask.to_dataset(name='barren_mask')
fmask.to_netcdf("barren_mask.nc")